<a href="https://colab.research.google.com/github/DavidGGs5/Fundamentos/blob/main/Unidad2%20/%20sistema_prestamos_modular%20v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Ingenieria en Sistemas Computacionales**

##**Fundamentos de programación**

### Unidad 2 - Listas, diccionarios y modulos

Docente: **José Gabriel Rodriguez Rivas**

Alumno: **David Leonardo Gómez Gaucín**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Colab Notebooks/Sistema de prestamos"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Sistema de prestamos


In [ ]:

%%writefile datos.py
alumnos = [{"nombre": "Juanito", "nume_control": 26040200,  "carrera": "Ingenieria en IA"},
            {"nombre": "Pedro","nume_control": 26040201, "carrera": "Ingenieria en sistemas"},
            {"nombre": "Jose","nume_control": 26040202, "carrera": "Ingenieria en IA"},
            {"nombre": "Maria","nume_control": 26040203, "carrera": "Ingenieria en informatica"},
            {"nombre": "Jimena","nume_control": 26040204, "carrera": "Ingenieria en IA"} ]
admins = [{"nombre": "Gabriel","nume_control": 123456, "ocupacion": "Profesor"},
          {"nombre": "Enrique","nume_control": 234567, "ocupacion": "Profesor"}]
computadoras = [
 {"id": "PC-LC1-01", "aula": "LC1", "software": "Python, VS Code", "estado": "Disponible", "usuario": "", "fecha_reserva": "" },
{"id": "PC-LC1-02", "aula": "LC1", "software": "Python, Java, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC1-03", "aula": "LC1", "software": "AUTOCAD, MATLAB", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC2-01", "aula": "LC2", "software": "Python, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC2-02", "aula": "LC2", "software": "Python, Java, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC2-03", "aula": "LC2", "software": "AUTOCAD, MATLAB", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC3-01", "aula": "LC3", "software": "Python, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC3-02", "aula": "LC3", "software": "Python, Java, VS Code", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC3-03", "aula": "LC3", "software": "AUTOCAD, MATLAB", "estado": "Disponible","usuario": "", "fecha_reserva": "" },
{"id": "PC-LC3-04", "aula": "LC3", "software": "Python, VS Code", "estado": "Mantenimiento","usuario": "", "fecha_reserva": "" }
]



historial = []
necesidades_software = []


Overwriting datos.py


In [ ]:
%%writefile persistencia.py
import json
import os
from datos import alumnos, admins, computadoras, historial, necesidades_software

RUTA = '/content/drive/MyDrive/prestamos_datos.json'

def cargar_datos():
    if not os.path.exists(RUTA):
        return  # primera vez: se queda con los datos iniciales
    with open(RUTA, "r", encoding="utf-8") as f:
        datos = json.load(f)
    alumnos[:] = datos["alumnos"]
    admins[:] = datos["admins"]
    computadoras[:] = datos["computadoras"]
    historial[:] = datos.get("historial", [])
    necesidades_software[:] = datos.get("necesidades_software", [])
def guardar_datos():
    datos = {"alumnos": alumnos, "admins": admins, "computadoras": computadoras,
             "historial": historial,
             "necesidades_software": necesidades_software}
    with open(RUTA, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)


Overwriting persistencia.py


In [ ]:
%%writefile equipos.py
from datos import computadoras, necesidades_software
from persistencia import guardar_datos
from datetime import datetime



def consultar_computadoras():
  print("\n--- COMPUTADORAS ---")
  for i in (computadoras):
    print(f"| Aula: {i["aula"]} | ID: {i["id"]} | Software: {i["software"]} | Estado: {i["estado"]} | Usuario: {i["usuario"]} | Fecha de reservación: {i["fecha_reserva"]}")
    print("---------------------")

def buscar_soft(num_control):
    programa = input("\nIngrese el software que desea buscar: ")
    print("\n--- RESULTADOS ---")
    encontrado = False
    for i in computadoras:
        if programa.lower() in i["software"].lower():
            print(f"ID: {i['id']} | Aula: {i['aula']} | Software: {i['software']} | Estado: {i['estado']}")
            encontrado = True
    if encontrado == False:
        print("No se encontraron computadoras con ese software.")
        propuesta = input(f"¿Quieres registrar {programa} como un software a futuro?"
                          f" Presiona S para aceptar: ")
        if propuesta.lower() == "s":
          necesidades_software.append({"usuario": num_control,
                                      "software": programa,
                                      "fecha": datetime.now().strftime("%d/%m/%Y %H:%M")})
          guardar_datos()
          print("El software se puso en lista de solcitado a espera de verificar")
          return
        else:
          print("No se registro nada")


def buscar_computadora_por_id(id_comp):
    for i in computadoras:
        if i["id"] == id_comp:
            return i
    return None


def buscar_pc_de_usuario(num_control):
    for pc in computadoras:
        if pc["usuario"] == num_control:
            return pc
    return None



def agreg_comp(num_control):
  if not es_admin(num_control):
        print("Solo un admin puede agregar computadoras.")
        return
  print("\n--- AGREGAR COMPUTADORA ---")
  id_computadora = input("ID de la computadora: ")
  if buscar_computadora_por_id(id_computadora) is not None:
      print("Error: Ya existe una computadora con ese ID.")
      return
  aula = input("Aula: ")
  aulas = {pc["aula"] for pc in computadoras}
  if aula not in aulas:
      print("Esta aula no existe")
      return
  soow = input("Software instalado: ")
  nueva_pc = {
      "id": id_computadora,
      "aula": aula,
      "software": soow,
      "estado": "Disponible",
      "usuario": "",
      "fecha_reserva": ""
  }
  computadoras.append(nueva_pc)
  computadoras.sort(key=lambda i: (i.get("aula", ""), i.get("id", "")))
  guardar_datos()
  print("\nComputadora agregada correctamente.")

Overwriting equipos.py


In [ ]:
%%writefile usuarios.py
from datos import alumnos, admins
from persistencia import guardar_datos



def pedir_num_control(mensaje="Ingresa tu numero de control: "):
    try:
        return int(input(mensaje))
    except ValueError:
        return None

def es_alumno(num_control):
    return any(a["nume_control"] == num_control for a in alumnos)

def es_admin(num_control):
    return any(a["nume_control"] == num_control for a in admins)

def existe_usuario(num_control):
    return es_alumno(num_control) or es_admin(num_control)


def agre_alu_o_admi():
    print("\n--- A quien vas a agregar ----")
    try:
        opcic = int(input("1 para agregar alumno, 2 para agregar admin, 3 para cancelar: "))
    except ValueError:
        print("Ingrese una opcion valida")
        return

    if opcic == 3:
        return
    if opcic not in (1, 2):
        print("Opcion no valida")
        return

    num = pedir_num_control("Numero de control: ")
    if num is None:
        print("El numero de control debe ser numerico.")
        return
    if existe_usuario(num):
        print("Ese numero de control ya existe.")
        return
    nombre = input("Nombre: ").strip()   # strip quita espacios sobrantes

    if opcic == 1:
        carrera = input("Carrera: ").strip()
        alumnos.append({"nombre": nombre, "nume_control": num, "carrera": carrera})
        print(f"El alumno {nombre} fue agregado con exito")
    elif opcic == 2:
        ocupacion = input("Ocupacion: ").strip()
        admins.append({"nombre": nombre, "nume_control": num, "ocupacion": ocupacion})
        print(f"El admin {nombre} fue agregado con exito")
    guardar_datos()

Overwriting usuarios.py


In [ ]:
%%writefile prestamos.py
from datetime import datetime, timedelta
from datos import computadoras, historial, necesidades_software
from persistencia import guardar_datos
from equipos import buscar_computadora_por_id, buscar_pc_de_usuario
from usuarios import es_alumno, es_admin



def solic_pres(num_control):
  pc_actual = buscar_pc_de_usuario(num_control)
  if pc_actual is not None and es_alumno(num_control):   # CAMBIO
    print(f"Ya tienes asignada la {pc_actual['id']}")
    return

  programa = input("\nIngrese el software que desea: ").strip().lower()
  if not programa:                        # evita que Enter vacio coincida con todo
    print("Debe escribir un programa.")
    return
  encontrado = False
  for i in computadoras:

    programas = []
    for s in i["software"].split(","):  #split transofrma los datos en forma de lista
      programas.append(s.strip().lower())

    if programa.lower() in programas:
      encontrado = True
      if i["estado"] == "Disponible":
        i["estado"] = "Prestada"
        i["usuario"] = num_control
        historial.append({"usuario": num_control,
                          "computadora": i["id"],
                          "software": programa,
                          "tipo": "Prestada",
                          "fecha_tramite": datetime.now().strftime("%d/%m/%Y %H:%M")})
        print("Préstamo registrado correctamente a favor del alumno.", num_control)
        print(f"Computadora: {i['id']} | Aula: {i['aula']}")
        guardar_datos()
        return
  if encontrado:
    print("Hay computadoras con ese software, pero ninguna esta disponible ahora.")
  else:
    print("No existe ninguna computadora con ese software.")
    propuesta = input(f"¿Quieres registrar {programa} como un software a futuro?"
                      f" Presiona S para aceptar o N para rechazar: ")
    if propuesta.lower() == "s":
      necesidades_software.append({"usuario": num_control,
                                  "software": programa,
                                  "fecha": datetime.now().strftime("%d/%m/%Y %H:%M")})
      guardar_datos()
      print("El software se puso en lista de solcitado a espera de verificar")
      return
    else:
      print("No se registro nada")


def devo_comp(num_control):
    print("\n--- DEVOLVER COMPUTADORA ---")
    id_computadora = input("Ingrese el ID de la computadora: ")
    i = buscar_computadora_por_id(id_computadora)
    if i is None:
        print("La computadora no existe.")
    elif i["estado"] == "Prestada" or i["estado"] == "Reservada":
      if i["usuario"] == num_control:
        i["estado"] = "Disponible"
        i["usuario"]= ""
        i["fecha_reserva"] = ""
        print("Computadora devuelta correctamente.")
        print(f"Computadora: {i['id']} | Aula: {i['aula']}")
        guardar_datos()
        return
      else:
        print("Esta computadora no fue prestada a este usuario")
    else:
        print("La computadora no esta registrada como prestada. Estado:", i["estado"])


def crea_reserv(num_control):
  usuario = buscar_pc_de_usuario(num_control)
  if usuario is not None and es_alumno(num_control):
    print("Este usuario ya tiene una computadora prestada o reservada")
    print(f"Ya tienes asignada la {usuario["id"]}")
    return
  else:
    print("\n--- CREAR RESERVACIÓN ---")
    hoy = datetime.now().date()
    limite_maximo = hoy + timedelta(days=3)
    print("Hoy es: ", hoy)
    print("")
    print("Puedes reservar hasta: ", limite_maximo)
    print("")


    programa = input("\nIngrese el software que desea: ").strip().lower()
  if not programa:                        # evita que Enter vacio coincida con todo
    print("Debe escribir un programa.")
    return
  encontrado = False
  for i in computadoras:

    programas = []
    for s in i["software"].split(","):  #split transofrma los datos en forma de lista
      programas.append(s.strip().lower())

    if programa.lower() in programas:
      encontrado = True
      if i["estado"] != "Disponible":
        continue

      fecha_solicitada = input("Ingresa la fecha que quieres reservar, usando el formato (DD/MM/YYYY): ")
      try:
        fecha_reservacion = datetime.strptime(fecha_solicitada, "%d/%m/%Y").date()
      except ValueError:
            print("Error: El formato de fecha no es válido. Debe ser DD/MM/YYYY.")
            return

      if hoy <= fecha_reservacion <= limite_maximo:
        i["estado"] = "Reservada"
        i["usuario"] = num_control
        i["fecha_reserva"] = fecha_reservacion.strftime("%d/%m/%Y")
        print("\n==============================================")
        print(" ¡Reservación registrada con éxito!")
        print(f" Computadora: {i['id']} (Aula: {i['aula']})")
        print(f" Reservada para el alumno: {num_control}")
        print(f" Fecha programada: {i['fecha_reserva']}")
        print("==============================================")
        historial.append({"usuario": num_control,
                          "computadora": i["id"],
                          "software": programa,
                          "tipo": "Reservada",
                          "fecha_tramite": datetime.now().strftime("%d/%m/%Y %H:%M"),
                          "fecha_programada": i["fecha_reserva"]
                          })
        guardar_datos()
        return
      elif fecha_reservacion < hoy: #Duda a preguntar
          print("No puedes solicitar la computadora para fechas pasadas.")
          return
      elif fecha_reservacion > limite_maximo:
          print("La fecha excede el límite permitido:", limite_maximo)
          return
  if encontrado:
    print("No hay computadoras con ese software disponibles actualmente")
  else:
    print("No existe ninguna computadora con el software")


def liberar_reservas_vencidas():
    hoy = datetime.now().date()
    for i in computadoras:
        if i["estado"] == "Reservada" and i["fecha_reserva"]:
            fecha = datetime.strptime(i["fecha_reserva"], "%d/%m/%Y").date()
            if fecha < hoy:
                i["estado"] = "Disponible"
                i["usuario"] = ""
                i["fecha_reserva"] = ""
                guardar_datos()

def historial_prest(num_control):
  print("\n--- HISTORIAL DE PRÉSTAMOS ---")
  control_admin = es_admin(num_control)
  varcontrol = False
  for t in reversed(historial):
    if control_admin or t["usuario"] == num_control:
      registro = (f"{t["fecha_tramite"]} | Usuario: {t["usuario"]} | Computadora: {t["computadora"]} | "
                  f" {t["software"]} | Tipo: {t["tipo"]}")
      if "fecha_programada" in t:
        registro += f" | Para el: {t['fecha_programada']}"
      print(registro)
      varcontrol = True
  if not varcontrol:
      print("No hay registros en el historial.")



Overwriting prestamos.py


In [ ]:
%%writefile estadisticas.py
from collections import Counter
from datos import admins, historial, necesidades_software, alumnos
from usuarios import es_alumno, es_admin

# Estadisticas por programa, y por carrera

def estats(conteo):
  popular = conteo.most_common()
  posicion = 1
  for p in popular:
    soft = p[0]
    veces = p[1]
    print(f"  {posicion}. {soft} — {veces} solicitud(es)")
    posicion += 1


def mostrar_estadisticas(num_control):
    if not es_admin(num_control):
        print("Solo un admin puede ver las estadísticas.")
        return

    print("\n--- ESTADÍSTICAS DEL HISTORIAL ---")
    if not historial:
        print("Aún no hay registros en el historial.")
        return
    else:
      print(f"Total de trámites: {len(historial)}")

    print("\n------ Estadisticas por programa ------")
    software = Counter()
    for t in historial:
      for s in t["software"].split(","):
        software[s.strip().lower()] += 1
    estats(software)

    print("\n------ Estadisticas por carrera ------")
    carrera_alum = {}
    for a in alumnos:
      numero = a["nume_control"]
      carrera = a["carrera"]
      carrera_alum[numero] = carrera
    can_car = Counter()
    for t in historial:
      carrera = carrera_alum.get(t["usuario"])
      if carrera:
        can_car[carrera] += 1
    if can_car:
      estats(can_car)
    else:
      print("Sin tramites de alumnos aun")

def software_necesidad(num_control):
  if not es_admin(num_control):
    print("Debes ser admin para tener acceso")
    return
  print("\n ------ NECESIDADES DE SOFTWARE NO EXISTENTE -----")
  if not necesidades_software:
    print("No hay registros aun")
    return
  conteo = Counter()
  for n in necesidades_software:
    nombre = n["software"].strip().lower()
    conteo[nombre] += 1
  popular = conteo.most_common()
  posicion = 1
  for p in popular:
    soft = p[0]
    veces = p[1]
    print(f"  {posicion}. {soft} — {veces} solicitud(es)")
    posicion += 1



Overwriting estadisticas.py


In [ ]:
%%writefile menu.py
from datos import alumnos, admins
from equipos import consultar_computadoras, buscar_soft, agreg_comp
from usuarios import agre_alu_o_admi, pedir_num_control, existe_usuario, es_admin
from prestamos import solic_pres, devo_comp, crea_reserv, historial_prest
from estadisticas import mostrar_estadisticas, software_necesidad


def sistema_de_prestamo (num_control):
  opcion = 0
  while True:
    print("\n==============================")
    print("      Sistema de prestamos  ")
    print("================================")
    print("1. Consultar computadoras y estados")
    print("2. Buscar por software")
    print("3. Solicitar prestamo")
    print("4. Devolver computadora")
    print("5. Crear reservacion")
    print("6. Agregar computadora")
    print("7. Agregar alumno o admin")
    print("8. Historial de prestamos")
    print("9. Estadisticas")
    print("10. Necesidades de software")
    print("11. Cambiar de cuenta")
    print("12. Salir")
    try:
      opcion = int(input("Seleccione una opcion: "))
    except ValueError:
      print("Error, debe ingresar un numero correspondiente al menu")
      continue

    if opcion == 1:
      consultar_computadoras()
    elif opcion == 2:
      buscar_soft(num_control)
    elif opcion == 3:
      solic_pres(num_control)
    elif opcion == 4:
      devo_comp(num_control)
    elif opcion == 5:
     crea_reserv(num_control)
    elif opcion == 6:
      agreg_comp()
    elif opcion == 7:
      agre_alu_o_admi()
    elif opcion == 8:
      historial_prest(num_control)
    elif opcion == 9:
      mostrar_estadisticas(num_control)
    elif opcion == 10:
      software_necesidad(num_control)
    elif opcion == 11:
      nuevo_control = pedir_num_control()
      if existe_usuario(nuevo_control):
        num_control = nuevo_control
      else:
        print("El alumno o admi no existe")
    elif opcion == 12:
      print(f"Saliendo del sistema... Hasta luego {num_control}")
      break
    else:
      print("Opcion no valida")

Overwriting menu.py


In [ ]:
%%writefile main.py
from persistencia import cargar_datos
from prestamos import liberar_reservas_vencidas
from menu import sistema_de_prestamo
from usuarios import pedir_num_control, existe_usuario


def main():

  cargar_datos()
  liberar_reservas_vencidas()


  num_control =  pedir_num_control()
  if existe_usuario(num_control):
    sistema_de_prestamo(num_control)
  else:
      print("El alumno o admi no existe")

if __name__ == "__main__":
    main()

Overwriting main.py


In [ ]:
from main import main
main()

Ingresa tu numero de control: 26040200

      Sistema de prestamos  
1. Consultar computadoras y estados
2. Buscar por software
3. Solicitar prestamo
4. Devolver computadora
5. Crear reservacion
6. Agregar computadora
7. Agregar alumno o admin
8. Historial de prestamos
9. Estadisticas
10. Necesidades de software
11. Cambiar de cuenta
12. Salir
Seleccione una opcion: 8

--- HISTORIAL DE PRÉSTAMOS ---
15/10/2026 02:45 | Usuario: 26040200 | Computadora: PC-LC1-01 |  python | Tipo: Reservada | Para el: 15/10/2026
15/10/2026 02:44 | Usuario: 26040200 | Computadora: PC-LC1-01 |  python | Tipo: Prestada

      Sistema de prestamos  
1. Consultar computadoras y estados
2. Buscar por software
3. Solicitar prestamo
4. Devolver computadora
5. Crear reservacion
6. Agregar computadora
7. Agregar alumno o admin
8. Historial de prestamos
9. Estadisticas
10. Necesidades de software
11. Cambiar de cuenta
12. Salir
Seleccione una opcion: 11
Ingresa tu numero de control: 123456

      Sistema de prestamo

##En el codigo de arriba tengo dividido en admin y alumno, cuando un alumno solicite su historial solo vera el suyo y cuando un admin lo solicite vera el de todos.



## Un admin puede pedir más de un prestamo a la vez mientras que un alumno no puede, por eso en estadisticas salen 5 solicitudes de python y 2 prestadas a ingenieria en IA (3 solicitudes de admin)

##Todo esta guardado en drive como base de datos, por eso salen datos que no se solicitaron en el momento (por eso tableau tiene 2 solicitudes, ya lo habia probado 1 vez)

